In [1]:
# Core imports
# Ensure `notebooks/` is on sys.path so `from utils...` works reliably.
import sys
from pathlib import Path

_here = Path.cwd()
if (_here / "utils").exists():
    # Running with cwd == notebooks/
    pass
elif (_here / "notebooks" / "utils").exists():
    # Running with cwd == repo root
    sys.path.insert(0, str(_here / "notebooks"))

import dspy
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
from openai import OpenAI
import os



In [2]:
# DSPy Configuration
lm = dspy.LM(
    model="openai/gpt-5.2", 
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)



### Core Policy Model

In [3]:
from utils.schemas import ExtractedPolicy

In [4]:
from utils.schemas import DocumentMetadata 


---

## 🧠 DSPy Signatures & Modules {#dspy}

Define DSPy signatures and custom modules for policy extraction, validation, and processing.

### Policy Extraction Signature

In [5]:
from utils.dspy_extraction import PolicyExtractionSignature, PolicyExtractor

In [6]:
# (Legacy strict validator removed)
# Individual validation uses the refined validator in `utils.dspy_validation`.
from utils.dspy_validation import PolicyValidationSignature, PolicyValidator, ValidationMetrics

In [7]:
from utils.docling_io import pdf_to_markdown

In [8]:
PDF_PATH = "../pdfs/Chicago-CAP-071822.pdf"

markdown_version = pdf_to_markdown(
    PDF_PATH,
    save_markdown_path="markdown_version_chicago.md",
)

print("Converted PDF to markdown:", PDF_PATH)
print("Markdown characters:", len(markdown_version))

2026-02-22 14:42:57,644 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-02-22 14:42:58,172 - INFO - Going to convert document batch...
2026-02-22 14:42:58,187 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-02-22 14:42:58,194 - INFO - Loading plugin 'docling_defaults'
2026-02-22 14:42:58,197 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-02-22 14:42:58,216 - INFO - Loading plugin 'docling_defaults'
2026-02-22 14:42:58,220 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-02-22 14:42:59,184 - INFO - Auto OCR model selected ocrmac.
2026-02-22 14:42:59,188 - INFO - Loading plugin 'docling_defaults'
2026-02-22 14:42:59,193 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2026-02-22 14:43:02,020 - INFO - Accelerator device: 'mps'
2026-02-22 14:43:03,604 - INFO - Loading plugin 'docling_defaul

Converted PDF to markdown: ../pdfs/Chicago-CAP-071822.pdf
Markdown characters: 256905


In [7]:
# notebooks/markdown_version_chicago.md take this and make it into a string

with open("markdown_version_chicago.md", "r") as file:
    markdown_version = file.read()


In [8]:
d_metadata = DocumentMetadata(
    country="USA",
    state_or_province="Illinois",
    city="Chicago"
)


In [9]:
lm = dspy.LM(
    model="openai/gpt-5.2", 
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)

In [10]:
policy_extractor = PolicyExtractor()
extracted_policies = policy_extractor(
    document_text=markdown_version,
    document_metadata=d_metadata
)


In [11]:
extracted_policies



[ExtractedPolicy(policy_statement='Invest $188 million in equity-focused climate interventions, including building retrofits, expansion of the air quality monitoring network, and planting over 75,000 trees.', verbatim_text="To kickstart the implementation of my administration's climate strategy, we are investing $188 million in equity-focused interventions, spanning building retrofits, the expansion of the air quality monitoring network, to the planting of over 75,000 trees.", policy_type='individual', parent_policy_name=None, section_header='Letter from the MAYOR', sector='Cross-sector', extraction_rationale='Explicit resource allocation ($188 million) tied to specific interventions; qualifies as a policy commitment.'),
 ExtractedPolicy(policy_statement='Plant 75,000 trees by 2026.', verbatim_text='Together with city departments and community partners, Chicago has the goal of planting 75,000 trees by 2026.', policy_type='individual', parent_policy_name=None, section_header='TREE EQUIT

In [35]:
import json

# Convert each Pydantic object to a dictionary
serializable_policies = [policy.model_dump() for policy in extracted_policies]

with open('extracted_policies.json', 'w', encoding='utf-8') as f:
    json.dump(serializable_policies, f, ensure_ascii=False, indent=2)
    

In [13]:
from utils.grounding import validate_grounding, validate_grounding_fragments

In [14]:
# Run grounding verification - try strict validation first
grounded_policies, rejected_log = validate_grounding(extracted_policies, markdown_version)

print(f"Extracted: {len(extracted_policies)} policies")
print(f"Strict validation passed: {len(grounded_policies)} policies")
print(f"Failed strict validation: {len(rejected_log)} policies")

# print(rejected_log)
# skip this for now

Extracted: 73 policies
Strict validation passed: 70 policies
Failed strict validation: 3 policies


In [15]:
from utils.clustering import cluster_policies, summarize_clusters

policy_clusters = cluster_policies(extracted_policies)
summarize_clusters(policy_clusters)

Total clusters:        25
  Parent+sub clusters: 14
  Individual clusters: 11
  Orphaned subs:       0

[Parent 1] [Pillar 1 | Strategy 1. Retrofit buildings]
  Pillar 1 Strategy 1: Retrofit buildings with specified targets across residentia
  └─ Sub 1: Retrofit residential buildings with 4 or fewer units: 20% by 2030 and 
  └─ Sub 2: Retrofit 20% of total 5+ unit residential buildings by 2030.
  └─ Sub 3: Retrofit 20% of total industrial buildings by 2030.
  └─ Sub 4: Retrofit 90% of total City-owned and sister agency-owned buildings by 
  └─ Sub 5: Retrofit 20% of total commercial buildings by 2035.
[Parent 2] [Pillar 1 | Strategy 2. Connect communities to renewable energy]
  Pillar 1 Strategy 2: Connect communities to renewable energy with targets for co
  └─ Sub 1: Install 5 megawatts of co-owned community solar projects by 2025.
  └─ Sub 2: Increase Chicago-based community renewables to 20 megawatts by 2025.
  └─ Sub 3: Increase community renewables subscriptions to achieve 25% su

In [16]:
from utils.clustering import clusters_to_records

# Build records and DataFrame
import pandas as pd

policy_records = clusters_to_records(policy_clusters)
df_policies = pd.DataFrame(policy_records)

# Reorder columns for readability
front_cols = [
    "cluster_id",
    "cluster_type",
    "role",
    "section_header",
    "sector",
    "policy_statement",
    "parent_statement",
    "verbatim_text",
    "extraction_rationale",
]
df_policies = df_policies[front_cols + [c for c in df_policies.columns if c not in front_cols]]

print(f"Total policy records: {len(df_policies)}")
print(f"Cluster breakdown:\n{df_policies.groupby(['cluster_type', 'role']).size().to_string()}")
df_policies.head(10)

Total policy records: 73
Cluster breakdown:
cluster_type      role      
individual        individual    11
parent_with_subs  parent        14
                  sub           48


,cluster_id,cluster_type,role,section_header,sector,policy_statement,parent_statement,verbatim_text,extraction_rationale,policy_type,parent_policy_name
0,0,individual,individual,Letter from the MAYOR,Cross-sector,Invest $188 million in equity-focused climate ...,None,To kickstart the implementation of my administ...,Explicit resource allocation ($188 million) ti...,individual,None
1,1,individual,individual,TREE EQUITY,AFOLU,"Plant 75,000 trees by 2026.",None,Together with city departments and community p...,"Quantified target with deadline (75,000 trees ...",individual,None
2,2,individual,individual,GHG REDUCTION TARGETS / ACCOUNTABILITY AND IMP...,Cross-sector,Reduce Chicago's GHG emissions by a minimum of...,None,The 2022 CAP aims to chart an equitable path t...,Quantifiable economy-wide emissions reduction ...,individual,None
3,3,individual,individual,The Role of Offsets,Cross-sector,Do not count carbon offset credits toward Chic...,None,Chicago will not count offset credits toward i...,Clear policy rule restricting use of offsets f...,individual,None
4,4,individual,individual,CLIMATE FINANCING AND DELIVERY CAPACITY,Buildings,Benchmarking Ordinance (2013) requiring covere...,None,"Benchmarking Ordinance, 2013: This ordinance s...",Binding mechanism (ordinance) with clear requi...,individual,None
5,5,individual,individual,CLIMATE FINANCING AND DELIVERY CAPACITY,Transport,EV Readiness Ordinance (2020) requiring specif...,None,"EV Readiness Ordinance, 2020: This ordinance r...",Binding mechanism (ordinance) with explicit pe...,individual,None
6,6,individual,individual,Chicago Recovery Plan funding overlap with Chi...,Buildings,Chicago Recovery Plan climate investment: Deca...,None,Decarbonize Affordable Multifamily Buildings |...,Resource allocation ($6M) with quantified deli...,individual,None
7,7,individual,individual,Chicago Recovery Plan funding overlap with Chi...,Buildings,Chicago Recovery Plan climate investment: Low-...,None,Low- or Moderate-Income (LMI) Housing Retrofit...,Resource allocation ($15M) with quantified del...,individual,None
8,8,individual,individual,Chicago Recovery Plan funding overlap with Chi...,AFOLU,Chicago Recovery Plan climate investment: Expa...,None,Expand Canopy Coverage | $46 million | Plant 7...,Budgeted program with quantified deliverable (...,individual,None
9,9,parent_with_subs,parent,Pillar 1 | Strategy 1. Retrofit buildings,Buildings,Pillar 1 Strategy 1: Retrofit buildings with s...,None,1 Increase access to utility savings and renew...,A named strategy with multiple lettered sub-ac...,parent,None


In [20]:
import json 
# Write the full DataFrame
df_policies.to_json("clustered_v4_policies.json", orient="records", indent=2)

print(f"Saved {len(df_policies)} policies to clustered_v4_policies.json")


Saved 76 policies to clustered_v4_policies.json


In [42]:
#now we take the individual policies and validate them

In [17]:
from utils.dspy_validation import PolicyValidationSignature, PolicyValidator, ValidationMetrics

In [23]:
# (Removed) DSPy was already configured above.
# Keeping a single LM config avoids accidental overrides.

In [19]:
from tqdm import tqdm
import pandas as pd

# Filter for individual policies
individual_policies = [r for r in policy_records if r.get("role") == "individual"]

validator = PolicyValidator()
validated_records = []

for record in tqdm(individual_policies):
    try:
        # Get result from the validator
        prediction = validator(policy_data=record)
        
        # Merge original data with all output fields from the prediction
        # dspy predictions can be converted to dicts directly
        full_record = {**record, **prediction.toDict()}
        validated_records.append(full_record)
        
    except Exception as e:
        print(f"Error: {e}")
        continue

df_validated = pd.DataFrame(validated_records)
df_validated.to_csv("validated_individual_policies.csv", index=False)
df_validated.head()

100%|██████████| 11/11 [01:25<00:00,  7.75s/it]


,policy_statement,verbatim_text,policy_type,parent_policy_name,section_header,sector,extraction_rationale,cluster_id,cluster_type,role,parent_statement,reasoning,validation_results
0,Invest $188 million in equity-focused climate ...,To kickstart the implementation of my administ...,individual,None,Letter from the MAYOR,Cross-sector,Explicit resource allocation ($188 million) ti...,0,individual,individual,None,The statement contains a specific funding amou...,has_quantifiable_target='Yes' has_timeline='No...
1,"Plant 75,000 trees by 2026.",Together with city departments and community p...,individual,None,TREE EQUITY,AFOLU,"Quantified target with deadline (75,000 trees ...",1,individual,individual,None,The verbatim text states a measurable outcome ...,has_quantifiable_target='Yes' has_timeline='Ye...
2,Reduce Chicago's GHG emissions by a minimum of...,The 2022 CAP aims to chart an equitable path t...,individual,None,GHG REDUCTION TARGETS / ACCOUNTABILITY AND IMP...,Cross-sector,Quantifiable economy-wide emissions reduction ...,2,individual,individual,None,"The statement contains a clear, measurable out...",has_quantifiable_target='Yes' has_timeline='Ye...
3,Do not count carbon offset credits toward Chic...,Chicago will not count offset credits toward i...,individual,None,The Role of Offsets,Cross-sector,Clear policy rule restricting use of offsets f...,3,individual,individual,None,This is a clear policy rule about accounting: ...,has_quantifiable_target='No' has_timeline='No'...
4,Benchmarking Ordinance (2013) requiring covere...,"Benchmarking Ordinance, 2013: This ordinance s...",individual,None,CLIMATE FINANCING AND DELIVERY CAPACITY,Buildings,Binding mechanism (ordinance) with clear requi...,4,individual,individual,None,The verbatim text describes an enforceable ord...,has_quantifiable_target='No' has_timeline='Yes...


In [22]:
len(df_validated)

11

In [ ]:
import pandas as pd

# 1. Convert the Pydantic objects into dictionaries
# (Using model_dump() for Pydantic v2, or dict() for Pydantic v1)
validation_data = df_validated['validation_results'].apply(
    lambda x: x.model_dump() if hasattr(x, 'model_dump') else x.dict()
)

# 2. Use json_normalize to flatten those dictionaries into a new DataFrame
df_metrics = pd.json_normalize(validation_data)

# 3. Join the new columns back to your original data and drop the "clump" column
df_final = pd.concat([df_validated.drop(columns=['validation_results']), df_metrics], axis=1)

# 4. Now you can index into them easily!
# Example: df_final['final_verdict'] or df_final[df_final['is_actionable'] == True]

# 5. Save to CSV (now with separate columns for each metric)
df_final.to_csv("validated_policies_indexed.csv", index=False)



,policy_statement,verbatim_text,policy_type,parent_policy_name,section_header,sector,extraction_rationale,cluster_id,cluster_type,role,...,has_quantifiable_target,has_timeline,has_binding_mechanism,has_spatial_specificity,weak_language_detected,strong_language_detected,validation_result,confidence_score,validation_reasoning,final_verdict
0,Invest $188 million in equity-focused climate ...,To kickstart the implementation of my administ...,individual,None,Letter from the MAYOR,Cross-sector,Explicit resource allocation ($188 million) ti...,0,individual,individual,...,Yes,No,No,No,No,Yes,NON-SOUND,0.40,"Quantifiable target: Present via ""planting of ...",False
1,"Plant 75,000 trees by 2026.",Together with city departments and community p...,individual,None,TREE EQUITY,AFOLU,"Quantified target with deadline (75,000 trees ...",1,individual,individual,...,Yes,Yes,No,Yes,Yes,No,NON-SOUND,0.78,"Quantifiable target: Yes — ""planting 75,000 tr...",False
2,Reduce Chicago's GHG emissions by a minimum of...,The 2022 CAP aims to chart an equitable path t...,individual,None,GHG REDUCTION TARGETS / ACCOUNTABILITY AND IMP...,Cross-sector,Quantifiable economy-wide emissions reduction ...,2,individual,individual,...,Yes,Yes,No,Yes,Yes,No,NON-SOUND,0.78,Quantifiable target: Yes — the text specifies ...,False
3,Do not count carbon offset credits toward Chic...,Chicago will not count offset credits toward i...,individual,None,The Role of Offsets,Cross-sector,Clear policy rule restricting use of offsets f...,3,individual,individual,...,No,No,No,Yes,No,Yes,NON-SOUND,0.40,"Quantifiable target: The text says, ""will not ...",False
4,Benchmarking Ordinance (2013) requiring covere...,"Benchmarking Ordinance, 2013: This ordinance s...",individual,None,CLIMATE FINANCING AND DELIVERY CAPACITY,Buildings,Binding mechanism (ordinance) with clear requi...,4,individual,individual,...,No,Yes,Yes,Yes,Yes,Yes,NON-SOUND,0.40,Quantifiable target (OUTCOME): No. The text re...,False


In [23]:
df_final.where(df_final['final_verdict'] == True)

,policy_statement,verbatim_text,policy_type,parent_policy_name,section_header,sector,extraction_rationale,cluster_id,cluster_type,role,...,has_quantifiable_target,has_timeline,has_binding_mechanism,has_spatial_specificity,weak_language_detected,strong_language_detected,validation_result,confidence_score,validation_reasoning,final_verdict
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Chicago Recovery Plan climate investment: Expa...,Expand Canopy Coverage | $46 million | Plant 7...,individual,None,Chicago Recovery Plan funding overlap with Chi...,AFOLU,Budgeted program with quantified deliverable (...,8.0,individual,individual,...,Yes,Yes,No,Yes,No,Yes,VALID,0.86,"Quantifiable target: Yes — the text states ""Pl...",True
9,"Enable 2,500 new public passenger electric veh...","Enable 2,500 new public passenger electric veh...",individual,None,Pillar 4 | Strategy 2. Enable building and per...,Transport,Quantified infrastructure target with deadline...,17.0,individual,individual,...,Yes,Yes,No,Yes,Yes,No,VALID,0.83,"Quantifiable target: '2,500 new public passeng...",True


---

## 📈 Results & Visualization {#results}

Final pipeline outputs, analysis, and data export.

In [24]:
# Summary statistics
print(f"📊 Pipeline Results Summary")
print(f"- Total policies extracted: {len(extracted_policies) if 'extracted_policies' in locals() else 'N/A'}")
print(f"- Policies after validation: {len(valid_policies) if 'valid_policies' in locals() else 'N/A'}")
print(f"- Policy clusters identified: {len(policy_clusters) if 'policy_clusters' in locals() else 'N/A'}")

# Export results
# validated_policies.to_csv('validated_policies.csv', index=False)
# policy_clusters.to_json('policy_clusters.json', orient='records')

📊 Pipeline Results Summary
- Total policies extracted: 73
- Policies after validation: N/A
- Policy clusters identified: 25


In [3]:
# (Optional) quick exports for debugging — safe to skip.
# NOTE: The final combined export already contains these rows.

import pandas as pd

parent_policies = []
sub_policies = []

for cluster in policy_clusters:
    if cluster.get("cluster_type") == "parent_with_subs":
        parent_policies.append(cluster["parent"].model_dump())
        for s in cluster.get("subs", []):
            sub_policies.append(s.model_dump())

pd.DataFrame(parent_policies).to_csv("parent_policies.csv", index=False)
pd.DataFrame(sub_policies).to_csv("sub_policies.csv", index=False)

print("✅ Wrote parent_policies.csv and sub_policies.csv")

NameError: name 'policy_clusters' is not defined

In [25]:
# Initiative validation (parent + subs)
# Produces an initiative-level table (`df_initiatives`) that the final export step
# uses to attach initiative metrics to parent/sub rows + write initiative traces.

from utils.initiative_validator import run_initiative_validation

# LLM call
df_initiatives = run_initiative_validation(policy_clusters, verbose=True)

# Side-effect export (optional)
df_initiatives.to_csv("validated_initiatives.csv", index=False)

df_initiatives.head()

Validating initiatives: 100%|██████████| 14/14 [03:59<00:00, 17.07s/it]


════════════════════════════════════════════════════════════
  INITIATIVE VALIDATION SUMMARY
════════════════════════════════════════════════════════════
  Total initiatives evaluated:  14
  SOUND:   6
  PARTIAL: 7
  WEAK:    1
  Final verdict = True:  13
  Avg confidence:        0.71
  Avg coverage:          0.75
  Avg coherence:         0.85
════════════════════════════════════════════════════════════



,initiative_name,parent_statement,sector,num_sub_actions,coverage_score,coverage_assessment,coherence_score,coherence_assessment,aggregate_measurability,has_implementation_pathway,...,weak_signals,strong_signals,initiative_result,confidence_score,initiative_reasoning,final_verdict,sub_assessments,subs_strong,subs_moderate,subs_weak
0,Pillar 1 | Strategy 1. Retrofit buildings,Pillar 1 Strategy 1: Retrofit buildings with s...,Buildings,5,0.90,The sub-actions collectively match the parent ...,0.95,The sub-actions are complementary and non-over...,Yes,Yes,...,"No binding mechanism stated, no explicit spati...",5 of 5 subs have numeric targets and deadline ...,SOUND,0.82,This initiative aims to increase access to uti...,True,[{'action_label': 'Action 1: Retrofit resident...,5,0,0
1,Pillar 1 | Strategy 2. Connect communities to ...,Pillar 1 Strategy 2: Connect communities to re...,Energy,3,0.85,The sub-actions directly operationalize the pa...,0.90,The three sub-actions are complementary: they ...,Yes,Yes,...,No binding mechanism (ordinance/regulation) st...,"All 3 subs have numeric targets (5 MW, 20 MW, ...",SOUND,0.86,This initiative aims to increase access to ren...,True,[{'action_label': 'Action 1: Install 5 megawat...,3,0,0
2,Pillar 2 | Strategy 1. Reduce waste and landfi...,Pillar 2 Strategy 1: Reduce waste and landfill...,Waste,6,0.85,The sub-actions collectively cover the parent ...,0.90,The sub-actions are complementary and organize...,Yes,Yes,...,"enable, implement strategies, no specified pol...","Community-wide organics program by 2025, 90% d...",SOUND,0.78,1) The initiative aims to reduce waste and lan...,True,[{'action_label': 'Action 1: Introduce communi...,3,1,2
3,"Pillar 3 | Strategy 1. Make walking, biking, o...","Pillar 3 Strategy 1: Make walking, biking, or ...",Transport,3,0.70,The sub-actions collectively support the paren...,0.85,The three sub-actions are complementary: expan...,Yes,Yes,...,"Expand (no miles/corridors specified), no dead...",30% increase in Divvy/shared micromobility tri...,PARTIAL,0.72,This initiative targets mode shift by making w...,True,[{'action_label': 'Action 1: Expand high-quali...,2,1,0
4,Pillar 3 | Strategy 2. Increase transit perfor...,Pillar 3 Strategy 2: Increase transit performa...,Transport,4,0.75,The sub-actions collectively match the parent ...,0.85,The sub-actions are complementary and mutually...,No,Yes,...,"No numeric outcome targets (e.g., ridership/VM...",All 4 sub-actions have explicit deadline years...,PARTIAL,0.60,This initiative aims to improve transit perfor...,True,[{'action_label': 'Action 1: Update land use p...,0,3,1


In [ ]:
# (Optional) Explore parent/sub initiative strengths
from pathlib import Path

import pandas as pd

csv_path = Path("parent_sub_initiatives.csv")
if not csv_path.exists():
    print("parent_sub_initiatives.csv not found. Run the initiative export step that generates it, or skip this cell.")
else:
    df_simple = pd.read_csv(csv_path)

    # View all parent-sub relationships
    display(df_simple.head(10))

    # Find all sub-initiatives for a specific parent
    building_subs = df_simple[df_simple["parent_initiative"] == "1. Retrofit buildings"]
    display(building_subs[["sub_initiative", "sub_strength"]])

    # Group by parent initiative to see summary
    parent_summary = df_simple.groupby("parent_initiative").agg(
        total_subs=("sub_initiative", "count"),
        parent_result=("parent_result", "first"),
        strong_subs=("sub_strength", lambda x: (x == "strong").sum()),
    )

    display(parent_summary)

,total_subs,parent_result,strong_subs
parent_initiative,,,
1. 100%clean renewable energy,3,SOUND,3
1. Collect relevant data,3,PARTIAL,0
"1. Makewalking, biking, or transit viable options for all trips",3,SOUND,2
1. Reduce waste and landfilling,6,SOUND,3
1. Retrofit buildings,5,SOUND,5
2. Connect communities to renewable energy,3,SOUND,3
2. Enable buildingand personal vehicle electrification,5,SOUND,4
2. Enable data-driven decision-making,5,PARTIAL,0
2. Increase transit performance and encourage equitable transit-oriented development,4,PARTIAL,0


---

## Final combined export (one table) + split traces

This section produces:
- One **combined table** containing **individual** + **parent/sub** policy rows.
- Two **separate trace outputs**:
  - **Initiative traces** (parent + sub assessments)
  - **Individual policy traces** (individual validation reasoning)

Run this section after you have generated `df_policies` (or `clustered_v4_policies.json`), `df_final` (or `validated_policies_indexed.csv`), and (optionally) `df_initiatives` (or `validated_initiatives.csv`).

In [26]:
from pathlib import Path

import pandas as pd

from utils.exports import (
    build_combined_policies_table,
    export_combined_table_and_traces,
)

# Guardrails: this is meant to run *after* the pipeline ran in memory
required = [
    ("df_policies", "Run clustering + clusters_to_records first"),
    ("df_final", "Run individual validation (df_validated -> df_final) first"),
    ("policy_clusters", "Run clustering first"),
]
for name, hint in required:
    if name not in globals() or globals()[name] is None:
        raise RuntimeError(f"Missing `{name}`. {hint}.")

# df_initiatives is optional (only needed to attach initiative metrics + initiative traces)
df_inits = df_initiatives if "df_initiatives" in globals() else pd.DataFrame()

combined = build_combined_policies_table(
    df_policies=df_policies,
    df_final_individual=df_final,
    df_initiatives=df_inits,
    policy_clusters=policy_clusters,
)

paths = export_combined_table_and_traces(
    combined=combined,
    df_initiatives=(df_inits if len(df_inits) else None),
    output_dir=Path("outputs/policy_pipeline_v4"),
)

print("✅ Exports written:")
for k, v in paths.items():
    print(f"- {k}: {v}")

combined.head(10)

✅ Exports written:
- combined_table: outputs/policy_pipeline_v4/all_policies_combined.csv
- initiative_traces: outputs/policy_pipeline_v4/traces/initiative_parent_sub_traces.jsonl
- individual_traces: outputs/policy_pipeline_v4/traces/individual_policy_traces.jsonl


,cluster_id,cluster_type,role,section_header,sector,policy_statement,parent_statement,verbatim_text,extraction_rationale,policy_type,...,sub_assessments,trace_type,trace_id,trace_path,sub_index,sub_action_label,sub_strength,sub_has_quantifiable_target,sub_has_timeline,sub_is_concrete_action
0,0,individual,individual,Letter from the MAYOR,Cross-sector,Invest $188 million in equity-focused climate ...,None,To kickstart the implementation of my administ...,Explicit resource allocation ($188 million) ti...,individual,...,NaN,individual,ind:0,None,NaN,NaN,NaN,NaN,NaN,NaN
1,1,individual,individual,TREE EQUITY,AFOLU,"Plant 75,000 trees by 2026.",None,Together with city departments and community p...,"Quantified target with deadline (75,000 trees ...",individual,...,NaN,individual,ind:1,None,NaN,NaN,NaN,NaN,NaN,NaN
2,2,individual,individual,GHG REDUCTION TARGETS / ACCOUNTABILITY AND IMP...,Cross-sector,Reduce Chicago's GHG emissions by a minimum of...,None,The 2022 CAP aims to chart an equitable path t...,Quantifiable economy-wide emissions reduction ...,individual,...,NaN,individual,ind:2,None,NaN,NaN,NaN,NaN,NaN,NaN
3,3,individual,individual,The Role of Offsets,Cross-sector,Do not count carbon offset credits toward Chic...,None,Chicago will not count offset credits toward i...,Clear policy rule restricting use of offsets f...,individual,...,NaN,individual,ind:3,None,NaN,NaN,NaN,NaN,NaN,NaN
4,4,individual,individual,CLIMATE FINANCING AND DELIVERY CAPACITY,Buildings,Benchmarking Ordinance (2013) requiring covere...,None,"Benchmarking Ordinance, 2013: This ordinance s...",Binding mechanism (ordinance) with clear requi...,individual,...,NaN,individual,ind:4,None,NaN,NaN,NaN,NaN,NaN,NaN
5,5,individual,individual,CLIMATE FINANCING AND DELIVERY CAPACITY,Transport,EV Readiness Ordinance (2020) requiring specif...,None,"EV Readiness Ordinance, 2020: This ordinance r...",Binding mechanism (ordinance) with explicit pe...,individual,...,NaN,individual,ind:5,None,NaN,NaN,NaN,NaN,NaN,NaN
6,6,individual,individual,Chicago Recovery Plan funding overlap with Chi...,Buildings,Chicago Recovery Plan climate investment: Deca...,None,Decarbonize Affordable Multifamily Buildings |...,Resource allocation ($6M) with quantified deli...,individual,...,NaN,individual,ind:6,None,NaN,NaN,NaN,NaN,NaN,NaN
7,7,individual,individual,Chicago Recovery Plan funding overlap with Chi...,Buildings,Chicago Recovery Plan climate investment: Low-...,None,Low- or Moderate-Income (LMI) Housing Retrofit...,Resource allocation ($15M) with quantified del...,individual,...,NaN,individual,ind:7,None,NaN,NaN,NaN,NaN,NaN,NaN
8,8,individual,individual,Chicago Recovery Plan funding overlap with Chi...,AFOLU,Chicago Recovery Plan climate investment: Expa...,None,Expand Canopy Coverage | $46 million | Plant 7...,Budgeted program with quantified deliverable (...,individual,...,NaN,individual,ind:8,None,NaN,NaN,NaN,NaN,NaN,NaN
9,9,parent_with_subs,parent,Pillar 1 | Strategy 1. Retrofit buildings,Buildings,Pillar 1 Strategy 1: Retrofit buildings with s...,None,1 Increase access to utility savings and renew...,A named strategy with multiple lettered sub-ac...,parent,...,[{'action_label': 'Action 1: Retrofit resident...,initiative,init:Pillar 1 | Strategy 1. Retrofit buildings...,None,NaN,NaN,NaN,NaN,NaN,NaN
